In [1]:
import warnings
warnings.filterwarnings("ignore")
from sklearn.ensemble import RandomForestClassifier
import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification

In [2]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [3]:
# Need to choose patient_id from OUS_D1 in response_OUS
data = list(OUS_D1['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D2 with response_OUS
clinical_train = pd.merge(OUS_D1, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS', 'LRC', 'event_LRC'])]

In [4]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [5]:
# Check null values in D1
clinical_train.isnull().sum().sum()

0

In [6]:
""" Takes too long time 
# Check collinearity for OUS data 
df = clinical_train

# Create a correlation matrix
correlation_matrix = df.corr()

plt.figure(figsize=(15, 12))

# Plot a heatmap
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)
plt.show()
""" 

" Takes too long time \n# Check collinearity for OUS data \ndf = clinical_train\n\n# Create a correlation matrix\ncorrelation_matrix = df.corr()\n\nplt.figure(figsize=(15, 12))\n\n# Plot a heatmap\nsns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)\nplt.show()\n"

### COX assumption in Train data

In [7]:
df = pd.DataFrame(clinical_train)

# Fit Cox proportional hazards model
cph = CoxPHFitter(penalizer=0.00001)
cph.fit(df, duration_col='DFS', event_col='event_DFS')

# Perform the proportional hazards assumption test
results = proportional_hazard_test(cph, df)
print(results)

<lifelines.CoxPHFitter: fitted with 139 total observations, 71 right-censored observations>

<lifelines.StatisticalResult: proportional_hazard_test>
    time_transform = rank
 null_distribution = chi squared
degrees_of_freedom = 1
             model = <lifelines.CoxPHFitter: fitted with 139 total observations, 71 right-censored observations>
         test_name = proportional_hazard_test

---
                test_statistic    p  -log2(p)
MTV                       0.00 0.95      0.08
SUVpeak                   0.29 0.59      0.76
TLG                       0.22 0.64      0.65
age                       1.60 0.21      2.28
cavum_oris                0.00 1.00      0.01
charlson                  0.07 0.79      0.34
female                    0.17 0.68      0.56
histgrade_high            1.26 0.26      1.93
hpv_related               4.31 0.04      4.72
hypopharynx               0.00 0.99      0.01
larynx                    0.00 0.98      0.02
oropharynx                0.00 0.99      0.02
pack_years                0.71 0.40      1.32
uicc8_III-IV              0.45 0.50      0.99


In [8]:
# Access the summary table
summary_table = results.summary

# Filter columns based on p-values lower than 0.05
significant_columns = summary_table[summary_table['p'] < 0.05].index

# Display significant columns
print("Columns with p-values < 0.05:")
print(significant_columns)

Columns with p-values < 0.05:
Index(['hpv_related'], dtype='object')


In [9]:
df = pd.DataFrame(clinical_train)

# Fit Cox proportional hazards model
cph = CoxPHFitter(penalizer=0.1)
cph.fit(df, duration_col='DFS', event_col='event_DFS')

# Perform the proportional hazards assumption test
results = proportional_hazard_test(cph, df)
print(results)

<lifelines.CoxPHFitter: fitted with 139 total observations, 71 right-censored observations>

<lifelines.StatisticalResult: proportional_hazard_test>
    time_transform = rank
 null_distribution = chi squared
degrees_of_freedom = 1
             model = <lifelines.CoxPHFitter: fitted with 139 total observations, 71 right-censored observations>
         test_name = proportional_hazard_test

---
                test_statistic    p  -log2(p)
MTV                       0.02 0.88      0.18
SUVpeak                   0.56 0.46      1.13
TLG                       0.06 0.81      0.30
age                       1.58 0.21      2.26
cavum_oris                0.08 0.77      0.37
charlson                  0.03 0.86      0.22
female                    0.35 0.56      0.85
histgrade_high            0.95 0.33      1.60
hpv_related               3.02 0.08      3.60
hypopharynx               0.10 0.75      0.41
larynx                    1.46 0.23      2.14
oropharynx                0.96 0.33      1.61
pack_years                0.39 0.53      0.92
uicc8_III-IV              0.08 0.78      0.36


In [10]:
# Access the summary table
summary_table = results.summary

# Filter columns based on p-values lower than 0.05
significant_columns = summary_table[summary_table['p'] < 0.05].index

# Display significant columns
print("Columns with p-values < 0.05:")
print(significant_columns)

Columns with p-values < 0.05:
Index([], dtype='object')


###### penalizer values essentially result in same result 

## Test dataset: MAASTRO 

In [11]:
(MAASTRO_D1['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [12]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [13]:
# need to choose patient_id from MAASTRO_D1 in response_MAASTRO
data = list(MAASTRO_D1['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 
response_MAASTRO

,patient_id,OS,OS_event,LRC,LRC_event,DFS,DFS_event
0,1,62.43,0.0,62.43,0.0,62.43,0.0
1,2,60.00,0.0,60.00,0.0,60.00,0.0
2,3,44.43,1.0,8.83,1.0,8.83,1.0
3,4,37.20,1.0,19.37,0.0,19.73,1.0
5,6,59.23,0.0,59.23,0.0,59.23,0.0
...,...,...,...,...,...,...,...
109,110,19.00,1.0,13.27,1.0,13.27,1.0
110,111,85.87,1.0,82.83,0.0,85.87,1.0
111,112,42.87,0.0,42.87,0.0,42.87,0.0
112,113,58.93,0.0,58.93,0.0,58.93,0.0


In [14]:
# Merge MAASTRO_D1 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D1, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'OS_event', 'LRC', 'LRC_event'])]
clinical_test

,patient_id,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG,DFS,DFS_event
0,1,55,0,0,1,0,0,1,1,1,0,0,15.438583,22.841,263.611623,62.43,0.0
1,2,55,0,0,1,0,0,0,0,0,20,1,8.829353,5.660,36.980700,60.00,0.0
2,3,55,0,0,1,0,0,0,0,1,6,1,13.476123,7.791,74.636342,8.83,1.0
3,4,61,1,0,0,0,1,1,0,1,45,1,8.632732,7.908,46.791979,19.73,1.0
4,6,70,0,0,1,0,0,1,1,1,59,0,9.783954,15.237,107.637514,59.23,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,110,66,1,0,0,0,1,0,0,0,55,1,31.338410,6.110,144.490782,13.27,1.0
95,111,63,0,0,0,0,1,0,0,1,174,1,13.041604,7.182,69.214868,85.87,1.0
96,112,63,0,0,1,0,0,1,1,1,0,1,8.944517,16.483,102.594274,42.87,0.0
97,113,54,0,0,1,0,0,1,1,0,0,0,14.184236,9.981,103.229492,58.93,0.0


In [15]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [16]:
# Some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG,DFS,DFS_event


In [17]:
# X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS'])]

# y 
y = clinical_train.loc[:, ['DFS', 'event_DFS']]

In [18]:
# Set lower, upper time point and times for IBS calculation later 
lower, upper = np.percentile(y['DFS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [19]:
# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

X_train:  (139, 14)
y_train:  (139,)


In [20]:
clinical_test.rename(columns = {'DFS_event' : 'event_DFS'}, inplace = True)

In [21]:
# Set X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'event_DFS'])]

# Set y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['DFS', 'event_DFS']]

# Change y_MAASTRO into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_DFS'], y_MAASTRO['DFS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

(99, 16)

In [22]:
# VIF dataframe 
vif_data = pd.DataFrame() 
vif_data["feature"] = X.columns 
  
# calculating VIF for each feature 
vif_data["VIF"] = [variance_inflation_factor(X.values, i) 
                          for i in range(len(X.columns))] 

vif_data

,feature,VIF
0,age,1.136852
1,female,1.197250
2,cavum_oris,7.993011
3,oropharynx,60.199385
4,hypopharynx,11.118710
5,larynx,14.183474
6,histgrade_high,1.150479
7,hpv_related,4.309385
8,charlson,1.302350
9,pack_years,1.647441


# Standardization

In [23]:
original_X = X.copy()

In [24]:
# Standardize the data 
## Save the column and index 
X_columns = X.columns 
X_index = X.index

## Standardize the data but not the categorical columns
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']
### Separate the categorical and non-categorical columns
X_categorical = X[categorical_columns]
X_numeric = X.drop(categorical_columns, axis=1)
X_numeric_columns = X_numeric.columns
X_numeric_index = X_numeric.index

### Standardize non-categorical and then concat with the categorical
scaler = StandardScaler()  
X_numeric_std = scaler.fit_transform(X_numeric)
X_numeric_std = pd.DataFrame(X_numeric_std, columns=X_numeric_columns, index=X_numeric_index)
X_std = pd.concat([X_categorical, X_numeric_std], axis=1)

## Sort the order of the columns as it was in the clinical train 
X_std = X_std[original_X.columns]

In [25]:
# Standardize X_MAASTRO 
MAASTRO_new = X_MAASTRO.copy()
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns

MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = scaler.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
MAASTRO_new = MAASTRO_new[original_X.columns]
MAASTRO_new_std = MAASTRO_new_std[MAASTRO_new.columns]

In [26]:
# Saving the data 
X_new = X 
X_new_std = X_std 
MAASTRO_new = X_MAASTRO 
MAASTRO_new_std = MAASTRO_new_std 

In [27]:
X_new

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG
0,54.238356,1,0,1,0,0,1,0.0,0,0.000000,0.0,14.473272,7.934,86.228420
1,54.539726,0,0,0,0,1,0,0.0,1,27.404795,0.0,5.044678,1.656,7.040100
2,59.019178,0,1,0,0,0,1,0.0,1,41.019178,1.0,7.839043,14.502,83.569669
3,70.726027,0,0,0,0,1,0,0.0,1,37.500000,0.0,2.880631,2.440,5.567091
4,67.865753,0,0,0,0,1,0,0.0,1,53.000000,0.0,5.402006,3.668,16.150550
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134,60.435616,0,0,1,0,0,1,1.0,0,0.000000,0.0,9.290139,3.650,26.280140
135,68.794521,0,0,1,0,0,1,1.0,0,0.000000,1.0,7.172883,18.967,101.754834
136,57.498630,0,0,1,0,0,1,1.0,1,39.498630,0.0,13.873187,6.370,66.273201
137,65.684932,0,0,1,0,0,1,1.0,1,71.527397,1.0,7.507419,12.443,71.832443


In [28]:
X_new_std

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG
0,-0.776210,1,0,1,0,0,1,0.0,0,-1.101176,0.0,0.646616,-0.296907,-0.179158
1,-0.737256,0,0,0,0,1,0,0.0,1,0.105775,0.0,-1.098689,-0.763433,-0.587333
2,-0.158251,0,1,0,0,0,1,0.0,1,0.705374,1.0,-0.581431,0.191169,-0.192863
3,1.354953,0,0,0,0,1,0,0.0,1,0.550384,0.0,-1.499271,-0.705173,-0.594926
4,0.985240,0,0,0,0,1,0,0.0,1,1.233028,0.0,-1.032545,-0.613919,-0.540373
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134,0.024835,0,0,1,0,0,1,1.0,0,-1.101176,0.0,-0.312822,-0.615257,-0.488161
135,1.105290,0,0,1,0,0,1,1.0,0,-1.101176,1.0,-0.704742,0.522969,-0.099128
136,-0.354794,0,0,1,0,0,1,1.0,1,0.638406,0.0,0.535536,-0.413130,-0.282017
137,0.703351,0,0,1,0,0,1,1.0,1,2.049005,1.0,-0.642817,0.038162,-0.253362


In [29]:
MAASTRO_new

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG
0,55,0,0,1,0,0,1,1,1,0,0,15.438583,22.841,263.611623
1,55,0,0,1,0,0,0,0,0,20,1,8.829353,5.660,36.980700
2,55,0,0,1,0,0,0,0,1,6,1,13.476123,7.791,74.636342
3,61,1,0,0,0,1,1,0,1,45,1,8.632732,7.908,46.791979
4,70,0,0,1,0,0,1,1,1,59,0,9.783954,15.237,107.637514
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,66,1,0,0,0,1,0,0,0,55,1,31.338410,6.110,144.490782
95,63,0,0,0,0,1,0,0,1,174,1,13.041604,7.182,69.214868
96,63,0,0,1,0,0,1,1,1,0,1,8.944517,16.483,102.594274
97,54,0,0,1,0,0,1,1,0,0,0,14.184236,9.981,103.229492


In [30]:
MAASTRO_new_std

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG
0,-0.677762,0,0,1,0,0,1,1,1,-1.101176,0,0.825303,0.810851,0.735160
1,-0.677762,0,0,1,0,0,0,0,0,-0.220344,1,-0.398117,-0.465891,-0.433005
2,-0.677762,0,0,1,0,0,0,0,1,-0.836927,1,0.462036,-0.307534,-0.238909
3,0.097786,1,0,0,0,1,1,0,1,0.880696,1,-0.434513,-0.298839,-0.382433
4,1.261108,0,0,1,0,0,1,1,1,1.497278,0,-0.221413,0.245788,-0.068806
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,0.744076,1,0,0,0,1,0,0,0,1.321112,1,3.768483,-0.432451,0.121154
95,0.356302,0,0,0,0,1,0,0,1,6.562062,1,0.381603,-0.352789,-0.266854
96,0.356302,0,0,1,0,0,1,1,1,-1.101176,1,-0.376799,0.338380,-0.094801
97,-0.807020,0,0,1,0,0,1,1,0,-1.101176,0,0.593113,-0.144792,-0.091527


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [31]:
# Setting the y format for skf below  
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-18 14:54:46,837] A new study created in memory with name: no-name-af5fee8b-78ba-4338-a224-b0dfdabd9f04


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.5697211155378487
Fold 2 C-index: 0.5426356589147286
Fold 3 C-index: 0.7617021276595745
Fold 4 C-index: 0.6806083650190115


[I 2024-04-18 14:54:47,625] A new study created in memory with name: no-name-a521de8b-3391-4d78-8d0c-46a133926e20


Fold 5 C-index: 0.630901287553648
[I 2024-04-18 14:54:47,560] Trial 0 finished with value: 0.6371137109369622 and parameters: {}. Best is trial 0 with value: 0.6371137109369622.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6371137109369622], datetime_start=datetime.datetime(2024, 4, 18, 14, 54, 47, 10222), datetime_complete=datetime.datetime(2024, 4, 18, 14, 54, 47, 559979), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6371137109369622


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.25671409350000196
Fold 2 IBS: 0.2656989046953461
Fold 3 IBS: 0.168829941254425
Fold 4 IBS: 0.2959795071877819
Fold 5 IBS: 0.21043512284090152
[I 2024-04-18 14:54:48,246] Trial 0 finished with value: 0.23953151389569127 and parameters: {}. Best is trial 0 with value: 0.23953151389569127.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.23953151389569127], datetime_start=datetime.datetime(2024, 4, 18, 14, 54, 47, 770980), datetime_complete=datetime.datetime(2024, 4, 18, 14, 54, 48, 246092), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.23953151389569127


In [32]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [33]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.637
train_ibs:  0.24


#### Test

In [34]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [35]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.563
IBS score: 0.291


In [36]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [37]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis - Ridge

#### Train

In [38]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-18 14:54:58,523] A new study created in memory with name: no-name-f14e65c3-a16f-42c8-bee3-5b47322d63f2


  0%|          | 0/1 [00:00<?, ?it/s]

[I 2024-04-18 14:54:58,774] A new study created in memory with name: no-name-08615c4d-ab7f-407f-a5ce-58ca9c968131


Fold 1 C-index: 0.6374501992031872
Fold 2 C-index: 0.5038759689922481
Fold 3 C-index: 0.5595744680851064
Fold 4 C-index: 0.48859315589353614
Fold 5 C-index: 0.6652360515021459
[I 2024-04-18 14:54:58,717] Trial 0 finished with value: 0.5709459687352447 and parameters: {}. Best is trial 0 with value: 0.5709459687352447.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.5709459687352447], datetime_start=datetime.datetime(2024, 4, 18, 14, 54, 58, 551214), datetime_complete=datetime.datetime(2024, 4, 18, 14, 54, 58, 716900), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.5709459687352447


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.24724709637048833
Fold 2 IBS: 0.2320398857267834
Fold 3 IBS: 0.22898186411767818
Fold 4 IBS: 0.24197478791371138
Fold 5 IBS: 0.22939558350833852
[I 2024-04-18 14:54:59,244] Trial 0 finished with value: 0.23592784352739998 and parameters: {}. Best is trial 0 with value: 0.23592784352739998.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.23592784352739998], datetime_start=datetime.datetime(2024, 4, 18, 14, 54, 58, 946721), datetime_complete=datetime.datetime(2024, 4, 18, 14, 54, 59, 243800), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.23592784352739998


In [39]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [40]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.571
train_ibs:  0.236


#### Test

In [41]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [42]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.532


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.229


In [43]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [44]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-18 14:55:02,203] A new study created in memory with name: no-name-74ab9955-04eb-4629-bf12-d411dc2fb323


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.5426356589147286
Fold 3 C-index: 0.7617021276595745
Fold 4 C-index: 0.6768060836501901


[I 2024-04-18 14:55:02,923] A new study created in memory with name: no-name-caa674f8-256d-4eee-9802-b3b4ab29a764


Fold 5 C-index: 0.630901287553648
[I 2024-04-18 14:55:02,857] Trial 0 finished with value: 0.635556441914194 and parameters: {}. Best is trial 0 with value: 0.635556441914194.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.635556441914194], datetime_start=datetime.datetime(2024, 4, 18, 14, 55, 2, 231877), datetime_complete=datetime.datetime(2024, 4, 18, 14, 55, 2, 857005), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.635556441914194


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.2612135612445686
Fold 2 IBS: 0.26335163677719536
Fold 3 IBS: 0.17031007633731787
Fold 4 IBS: 0.29553230790913454
Fold 5 IBS: 0.21015944157044716
[I 2024-04-18 14:55:03,939] Trial 0 finished with value: 0.24011340476773274 and parameters: {}. Best is trial 0 with value: 0.24011340476773274.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.24011340476773274], datetime_start=datetime.datetime(2024, 4, 18, 14, 55, 3, 20064), datetime_complete=datetime.datetime(2024, 4, 18, 14, 55, 3, 938929), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.24011340476773274


In [45]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [46]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.636
train_ibs:  0.24


#### Test

In [47]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [48]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.566


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.288


In [49]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [50]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-18 14:55:04,846] A new study created in memory with name: no-name-fbfbb8d4-b39e-40ef-8cc1-60d3e26342d5


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.5426356589147286
Fold 3 C-index: 0.7617021276595745
Fold 4 C-index: 0.6768060836501901
Fold 5 C-index: 0.630901287553648
[I 2024-04-18 14:55:05,799] Trial 0 finished with value: 0.635556441914194 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.635556441914194.
Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.5426356589147286
Fold 3 C-index: 0.7617021276595745
Fold 4 C-index: 0.6806083650190115
Fold 5 C-index: 0.6351931330472103
[I 2024-04-18 14:55:06,356] Trial 1 finished with value: 0.6371752672866707 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.6371752672866707.
Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.5426356589147286
Fold 3 C-index: 0.7617021276595745
Fold 4 C-index: 0.6806083650190115
Fold 5 C-index: 0.6394849785407726
[I 2024-04-18 14:55:06,993] Trial 2 finished with value: 0.6380336363853832 and parameters: {'l1_ratio': 0.22692876841884668}. Be

Fold 2 C-index: 0.5426356589147286
Fold 3 C-index: 0.7617021276595745
Fold 4 C-index: 0.6806083650190115
Fold 5 C-index: 0.6351931330472103
[I 2024-04-18 14:55:20,255] Trial 24 finished with value: 0.6371752672866707 and parameters: {'l1_ratio': 0.3752730935447346}. Best is trial 2 with value: 0.6380336363853832.
Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.5426356589147286
Fold 3 C-index: 0.7617021276595745
Fold 4 C-index: 0.6806083650190115
Fold 5 C-index: 0.6351931330472103
[I 2024-04-18 14:55:20,574] Trial 25 finished with value: 0.6371752672866707 and parameters: {'l1_ratio': 0.26085123056535237}. Best is trial 2 with value: 0.6380336363853832.
Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.5426356589147286
Fold 3 C-index: 0.7617021276595745
Fold 4 C-index: 0.6806083650190115
Fold 5 C-index: 0.6351931330472103
[I 2024-04-18 14:55:21,098] Trial 26 finished with value: 0.6371752672866707 and parameters: {'l1_ratio': 0.43696099753129897}. Best is trial 2 with value: 0.6

Fold 2 C-index: 0.49806201550387597
Fold 3 C-index: 0.5553191489361702
Fold 4 C-index: 0.5019011406844106
Fold 5 C-index: 0.6609442060085837
[I 2024-04-18 14:55:33,458] Trial 48 finished with value: 0.5683449038202336 and parameters: {'l1_ratio': 0.04487776284637307}. Best is trial 2 with value: 0.6380336363853832.
Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.5426356589147286
Fold 3 C-index: 0.7617021276595745
Fold 4 C-index: 0.6806083650190115
Fold 5 C-index: 0.6351931330472103
[I 2024-04-18 14:55:34,218] Trial 49 finished with value: 0.6371752672866707 and parameters: {'l1_ratio': 0.2881440462226219}. Best is trial 2 with value: 0.6380336363853832.
Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.5426356589147286
Fold 3 C-index: 0.7617021276595745
Fold 4 C-index: 0.6806083650190115
Fold 5 C-index: 0.6351931330472103
[I 2024-04-18 14:55:34,951] Trial 50 finished with value: 0.6371752672866707 and parameters: {'l1_ratio': 0.41653985858071785}. Best is trial 2 with value: 0.

...

  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 IBS: 0.25928666566134556
Fold 2 IBS: 0.2633235500953682
Fold 3 IBS: 0.1703452680482903
Fold 4 IBS: 0.2954820759487805
Fold 5 IBS: 0.2100027239709883
[I 2024-04-18 14:56:04,040] Trial 0 finished with value: 0.23968805674495455 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.23968805674495455.
Fold 1 IBS: 0.2584786536006208
Fold 2 IBS: 0.26323896642742395
Fold 3 IBS: 0.17025549969200326
Fold 4 IBS: 0.29530473742584673
Fold 5 IBS: 0.20967534920375316
[I 2024-04-18 14:56:04,982] Trial 1 finished with value: 0.23939064126992954 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.23939064126992954.
Fold 1 IBS: 0.2583088882294021
Fold 2 IBS: 0.2631695969400213
Fold 3 IBS: 0.1701909536443265
Fold 4 IBS: 0.2952670408466882
Fold 5 IBS: 0.20960920881776335
[I 2024-04-18 14:56:05,763] Trial 2 finished with value: 0.2393091376956403 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 2 with value: 0.2393091376956403.
F

Fold 1 IBS: 0.2581741896656467
Fold 2 IBS: 0.23423392261218143
Fold 3 IBS: 0.17019839315276927
Fold 4 IBS: 0.29521831555599
Fold 5 IBS: 0.20960746005873962
[I 2024-04-18 14:56:18,726] Trial 25 finished with value: 0.2334864562090654 and parameters: {'l1_ratio': 0.1947967380354304}. Best is trial 25 with value: 0.2334864562090654.
Fold 1 IBS: 0.2581804446131643
Fold 2 IBS: 0.26312711872028444
Fold 3 IBS: 0.17016174202858292
Fold 4 IBS: 0.2952515096149463
Fold 5 IBS: 0.20957188290681883
[I 2024-04-18 14:56:19,653] Trial 26 finished with value: 0.23925853957675933 and parameters: {'l1_ratio': 0.20309774849254153}. Best is trial 25 with value: 0.2334864562090654.
Fold 1 IBS: 0.2586569332512195
Fold 2 IBS: 0.26321104962903547
Fold 3 IBS: 0.17023659190799084
Fold 4 IBS: 0.2953030694659617
Fold 5 IBS: 0.20971134084981743
[I 2024-04-18 14:56:20,494] Trial 27 finished with value: 0.23942379702080502 and parameters: {'l1_ratio': 0.3465220894892909}. Best is trial 25 with value: 0.233486456209065

Fold 1 IBS: 0.2588397617421469
Fold 2 IBS: 0.2631556097356599
Fold 3 IBS: 0.1702461973514651
Fold 4 IBS: 0.29536682726214786
Fold 5 IBS: 0.20979634151542792
[I 2024-04-18 14:56:33,685] Trial 50 finished with value: 0.23948094752136956 and parameters: {'l1_ratio': 0.41653985858071785}. Best is trial 25 with value: 0.2334864562090654.
Fold 1 IBS: 0.24700943918161714
Fold 2 IBS: 0.23211153171381074
Fold 3 IBS: 0.2287483556972372
Fold 4 IBS: 0.24295867311349534
Fold 5 IBS: 0.22883924884787327
[I 2024-04-18 14:56:34,185] Trial 51 finished with value: 0.2359334497108067 and parameters: {'l1_ratio': 0.006012844952470507}. Best is trial 25 with value: 0.2334864562090654.
Fold 1 IBS: 0.25776772949132926
Fold 2 IBS: 0.23341820114233583
Fold 3 IBS: 0.22558059996519
Fold 4 IBS: 0.29517332645907624
Fold 5 IBS: 0.22248441822949994
[I 2024-04-18 14:56:34,626] Trial 52 finished with value: 0.24688485505748625 and parameters: {'l1_ratio': 0.11725216969950066}. Best is trial 25 with value: 0.23348645620

Fold 1 IBS: 0.24584401855708746
Fold 2 IBS: 0.2325414275047241
Fold 3 IBS: 0.22751159212227245
Fold 4 IBS: 0.24840692200101858
Fold 5 IBS: 0.22611722092615838
[I 2024-04-18 14:56:44,114] Trial 75 finished with value: 0.23608423622225222 and parameters: {'l1_ratio': 0.041860881790211954}. Best is trial 25 with value: 0.2334864562090654.
Fold 1 IBS: 0.2575693918073802
Fold 2 IBS: 0.23318555218134743
Fold 3 IBS: 0.2260348499439707
Fold 4 IBS: 0.2548199486345508
Fold 5 IBS: 0.22328245681576514
[I 2024-04-18 14:56:44,450] Trial 76 finished with value: 0.2389784398766029 and parameters: {'l1_ratio': 0.09666857918599614}. Best is trial 25 with value: 0.2334864562090654.
Fold 1 IBS: 0.24621378571183025
Fold 2 IBS: 0.2323893088410591
Fold 3 IBS: 0.2279206974111101
Fold 4 IBS: 0.24657993346599297
Fold 5 IBS: 0.22697836138616592
[I 2024-04-18 14:56:44,694] Trial 77 finished with value: 0.23601641736323167 and parameters: {'l1_ratio': 0.029179515489863375}. Best is trial 25 with value: 0.233486456

In [51]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [52]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.638
train_ibs:  0.233


#### Test

In [53]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [54]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.22692876841884668)

test_cindex : 0.524


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.1947967380354304)

test_ibs:  0.235


In [55]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [56]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-18 14:56:54,453] A new study created in memory with name: no-name-92426e07-506a-49c7-957b-aac296c21a18


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5737051792828686
Fold 2 C-index: 0.6201550387596899
Fold 3 C-index: 0.625531914893617
Fold 4 C-index: 0.6673003802281369
Fold 5 C-index: 0.6072961373390557
[I 2024-04-18 14:57:01,043] Trial 0 finished with value: 0.6187977301006736 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.6187977301006736.
Fold 1 C-index: 0.5737051792828686
Fold 2 C-index: 0.6472868217054264
Fold 3 C-index: 0.6723404255319149
Fold 4 C-index: 0.6159695817490495
Fold 5 C-index: 0.6094420600858369
[I 2024-04-18 14:57:05,171] Trial 1 finished with value: 0.6237488136710192 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, '

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-18 14:57:51,224] Trial 16 finished with value: 0.5 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 5, 'min_samples_leaf': 16, 'max_depth': 1, 'n_estimators': 4, 'oob_score': True, 'max_samples': 0.44827686549014745, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.37909912926450273, 'warm_start': True}. Best is trial 15 with value: 0.6815364844139735.
Fold 1 C-index: 0.5697211155378487
Fold 2 C-index: 0.6937984496124031
Fold 3 C-index: 0.7914893617021277
Fold 4 C-index: 0.7167300380228137
Fold 5 C-index: 0.6759656652360515
[I 2024-04-18 14:57:52,323] Trial 17 finished with value: 0.6895409260222489 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 8, 'min_samples_leaf': 16, 'max_depth': 1, 'n_estimators': 83, 'oob_score': True, 'max_samples': 0.9817072304699028, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.2138706147057923, 'warm_start': Tru

Fold 1 C-index: 0.5617529880478087
Fold 2 C-index: 0.7093023255813954
Fold 3 C-index: 0.7957446808510639
Fold 4 C-index: 0.7338403041825095
Fold 5 C-index: 0.6952789699570815
[I 2024-04-18 14:58:24,265] Trial 31 finished with value: 0.6991838537239718 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 19, 'min_samples_leaf': 14, 'max_depth': 15, 'n_estimators': 498, 'oob_score': True, 'max_samples': 0.7425407774406056, 'max_features': None, 'min_weight_fraction_leaf': 0.06755614270828253, 'warm_start': True}. Best is trial 24 with value: 0.7028777296239799.
Fold 1 C-index: 0.5418326693227091
Fold 2 C-index: 0.7325581395348837
Fold 3 C-index: 0.8042553191489362
Fold 4 C-index: 0.7376425855513308
Fold 5 C-index: 0.7510729613733905
[I 2024-04-18 14:58:29,143] Trial 32 finished with value: 0.71347233498625 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 18, 'min_samples_leaf': 11, 'max_depth': 14, 'n_estimators': 482, 'oob_score': True, 'max_samples': 0.7213418356032232,

...

Fold 1 IBS: 0.2370347091364531
Fold 2 IBS: 0.21865616933268361
Fold 3 IBS: 0.2136904590169095
Fold 4 IBS: 0.2330616724900941
Fold 5 IBS: 0.21522492222501335
[I 2024-04-18 15:03:21,541] Trial 32 finished with value: 0.22353358644023075 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 18, 'min_samples_leaf': 4, 'max_depth': 13, 'n_estimators': 310, 'oob_score': True, 'max_samples': 0.8040571874379057, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.18786523282060735}. Best is trial 32 with value: 0.22353358644023075.
Fold 1 IBS: 0.2369596498960518
Fold 2 IBS: 0.21919381104819285
Fold 3 IBS: 0.21367435784125732
Fold 4 IBS: 0.2331924481484345
Fold 5 IBS: 0.21510595787220624
[I 2024-04-18 15:03:27,377] Trial 33 finished with value: 0.22362524496122854 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 19, 'min_samples_leaf': 4, 'max_depth': 13, 'n_estimators': 309, 'oob_score': True, 'max_samples': 0.7910445247744765, 'max_features': 'sqrt', 'min_weight_fraction_leaf

Fold 5 IBS: 0.21452585999296125
[I 2024-04-18 15:05:22,695] Trial 47 finished with value: 0.22331758578489697 and parameters: {'min_samples_split': 11, 'max_leaf_nodes': 18, 'min_samples_leaf': 2, 'max_depth': 9, 'n_estimators': 332, 'oob_score': True, 'max_samples': 0.8648447148686852, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.19908924320632904}. Best is trial 42 with value: 0.22312152539133093.
Fold 1 IBS: 0.23571677073756844
Fold 2 IBS: 0.21862106487993588
Fold 3 IBS: 0.21456356649548577
Fold 4 IBS: 0.23300957311771886
Fold 5 IBS: 0.2147097776323396
[I 2024-04-18 15:05:28,943] Trial 48 finished with value: 0.2233241505726097 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 18, 'min_samples_leaf': 5, 'max_depth': 9, 'n_estimators': 339, 'oob_score': True, 'max_samples': 0.85048001681846, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.21469064698337334}. Best is trial 42 with value: 0.22312152539133093.
Fold 1 IBS: 0.23559067856645097
Fold 2 IBS: 0.218738

Fold 1 IBS: 0.24251844789708726
Fold 2 IBS: 0.2210411248095999
Fold 3 IBS: 0.2174884582534811
Fold 4 IBS: 0.23327796476633547
Fold 5 IBS: 0.21325865662478627
[I 2024-04-18 15:07:00,107] Trial 63 finished with value: 0.225516930470258 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 14, 'min_samples_leaf': 5, 'max_depth': 11, 'n_estimators': 440, 'oob_score': True, 'max_samples': 0.8434390256503708, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.13262761517233745}. Best is trial 42 with value: 0.22312152539133093.
Fold 1 IBS: 0.24154578994370174
Fold 2 IBS: 0.22210454661611326
Fold 3 IBS: 0.21614753754341642
Fold 4 IBS: 0.23303912664616094
Fold 5 IBS: 0.21337979688122055
[I 2024-04-18 15:07:07,098] Trial 64 finished with value: 0.2252433595261226 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 14, 'min_samples_leaf': 3, 'max_depth': 10, 'n_estimators': 421, 'oob_score': True, 'max_samples': 0.9071936553420485, 'max_features': 'log2', 'min_weight_fraction_le

Fold 5 IBS: 0.21423498837277127
[I 2024-04-18 15:08:41,346] Trial 78 finished with value: 0.22373591837101642 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 17, 'min_samples_leaf': 7, 'max_depth': 10, 'n_estimators': 346, 'oob_score': True, 'max_samples': 0.8854068847594949, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.23765891161800223}. Best is trial 42 with value: 0.22312152539133093.
Fold 1 IBS: 0.23570867974207424
Fold 2 IBS: 0.22060193373952594
Fold 3 IBS: 0.2144464193518562
Fold 4 IBS: 0.23476296957407278
Fold 5 IBS: 0.21578644913062256
[I 2024-04-18 15:08:46,084] Trial 79 finished with value: 0.22426129030763034 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 13, 'min_samples_leaf': 5, 'max_depth': 5, 'n_estimators': 291, 'oob_score': True, 'max_samples': 0.9991782770608287, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.2796131771286568}. Best is trial 42 with value: 0.22312152539133093.
Fold 1 IBS: 0.25860616056552466
Fold 2 IBS: 0.247

Fold 1 IBS: 0.2364707434109151
Fold 2 IBS: 0.21910149125141531
Fold 3 IBS: 0.21420284080337593
Fold 4 IBS: 0.23438012728527638
Fold 5 IBS: 0.21479897289302796
[I 2024-04-18 15:10:32,950] Trial 94 finished with value: 0.22379083512880213 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 18, 'min_samples_leaf': 2, 'max_depth': 6, 'n_estimators': 430, 'oob_score': True, 'max_samples': 0.892312681224388, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.2167154554420358}. Best is trial 84 with value: 0.22302339534037285.
Fold 1 IBS: 0.23950861114017571
Fold 2 IBS: 0.22006606579763324
Fold 3 IBS: 0.2159169717512573
Fold 4 IBS: 0.2336883258166932
Fold 5 IBS: 0.21371153072549798
[I 2024-04-18 15:10:40,640] Trial 95 finished with value: 0.22457830104625148 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 19, 'min_samples_leaf': 4, 'max_depth': 9, 'n_estimators': 481, 'oob_score': True, 'max_samples': 0.8276360496389354, 'max_features': 'log2', 'min_weight_fraction_leaf'

In [57]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [58]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.76
train_ibs:  0.223


#### Test

In [59]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

In [60]:
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])
 
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=5, max_leaf_nodes=8,
                     max_samples=0.9313711932374781, min_samples_split=12,
                     min_weight_fraction_leaf=0.011633867937315842,
                     n_estimators=214, random_state=123, warm_start=True)

test_cindex:  0.633


RandomSurvivalForest(max_depth=6, max_features='log2', max_leaf_nodes=20,
                     max_samples=0.8096737689830545, min_samples_leaf=2,
                     min_samples_split=11,
                     min_weight_fraction_leaf=0.19628098798162147,
                     n_estimators=453, oob_score=True, random_state=123)

test_ibs:  0.216


In [61]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [62]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [63]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-18 15:11:10,019] A new study created in memory with name: no-name-f1f54b3e-d4d5-4893-9d83-9134843f50a9


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5856573705179283
Fold 2 C-index: 0.689922480620155
Fold 3 C-index: 0.7489361702127659
Fold 4 C-index: 0.7110266159695817
Fold 5 C-index: 0.6695278969957081
[I 2024-04-18 15:11:11,281] Trial 0 finished with value: 0.6810141068632278 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.6810141068632278.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-18 15:11:13,531] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531}

Fold 1 C-index: 0.6294820717131474
Fold 2 C-index: 0.6627906976744186
Fold 3 C-index: 0.7361702127659574
Fold 4 C-index: 0.6825095057034221
Fold 5 C-index: 0.648068669527897
[I 2024-04-18 15:11:47,088] Trial 16 finished with value: 0.6718042314769685 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 17, 'min_samples_leaf': 5, 'max_depth': 20, 'n_estimators': 499, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.6535355702555379, 'min_weight_fraction_leaf': 0.1906011147608997}. Best is trial 12 with value: 0.6867027610976928.
Fold 1 C-index: 0.6115537848605578
Fold 2 C-index: 0.6453488372093024
Fold 3 C-index: 0.7404255319148936
Fold 4 C-index: 0.6825095057034221
Fold 5 C-index: 0.6566523605150214
[I 2024-04-18 15:11:48,375] Trial 17 finished with value: 0.6672980040406395 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 10, 'min_samples_leaf': 8, 'max_depth': 10, 'n_estimators': 384, 'oob_score': False, 'warm_start': True, 'max_featur

Fold 1 C-index: 0.5418326693227091
Fold 2 C-index: 0.7015503875968992
Fold 3 C-index: 0.774468085106383
Fold 4 C-index: 0.7946768060836502
Fold 5 C-index: 0.7682403433476395
[I 2024-04-18 15:12:09,542] Trial 31 finished with value: 0.7161536582914563 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 16, 'min_samples_leaf': 1, 'max_depth': 20, 'n_estimators': 207, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.9402340730195752, 'min_weight_fraction_leaf': 0.07589392630375186}. Best is trial 28 with value: 0.7175262582650991.
Fold 1 C-index: 0.5617529880478087
Fold 2 C-index: 0.689922480620155
Fold 3 C-index: 0.7702127659574468
Fold 4 C-index: 0.7376425855513308
Fold 5 C-index: 0.721030042918455
[I 2024-04-18 15:12:10,584] Trial 32 finished with value: 0.6961121726190391 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 15, 'min_samples_leaf': 1, 'max_depth': 20, 'n_estimators': 207, 'oob_score': False, 'warm_start': True, 'max_features':

...

Fold 1 IBS: 0.24654710532719232
Fold 2 IBS: 0.23226075421028072
Fold 3 IBS: 0.22941657391206893
Fold 4 IBS: 0.24156072638580853
Fold 5 IBS: 0.23017807225897458
[I 2024-04-18 15:15:46,952] Trial 30 finished with value: 0.235992646418865 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 6, 'min_samples_leaf': 12, 'max_depth': 15, 'n_estimators': 225, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.46244718326068934, 'min_weight_fraction_leaf': 0.23846812660748434}. Best is trial 6 with value: 0.21817221600733724.
Fold 1 IBS: 0.23751141455085345
Fold 2 IBS: 0.2151581315756588
Fold 3 IBS: 0.20565940925991258
Fold 4 IBS: 0.22542469872025206
Fold 5 IBS: 0.21030193870732267
[I 2024-04-18 15:15:52,593] Trial 31 finished with value: 0.2188111185627999 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 4, 'min_samples_leaf': 3, 'max_depth': 9, 'n_estimators': 500, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.5

Fold 1 IBS: 0.23579057506686668
Fold 2 IBS: 0.21781214726067985
Fold 3 IBS: 0.2066254011517932
Fold 4 IBS: 0.22753811203645144
Fold 5 IBS: 0.2131782901601682
[I 2024-04-18 15:17:00,604] Trial 45 finished with value: 0.22018890513519188 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 9, 'min_samples_leaf': 7, 'max_depth': 5, 'n_estimators': 367, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.7526038702156265, 'min_weight_fraction_leaf': 0.19723066794982302}. Best is trial 6 with value: 0.21817221600733724.
Fold 1 IBS: 0.23864304620933718
Fold 2 IBS: 0.2216376067444548
Fold 3 IBS: 0.21798786862601383
Fold 4 IBS: 0.2317000936375164
Fold 5 IBS: 0.21855619191514636
[I 2024-04-18 15:17:05,258] Trial 46 finished with value: 0.2257049614264937 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 10, 'min_samples_leaf': 9, 'max_depth': 8, 'n_estimators': 306, 'oob_score': True, 'warm_start': True, 'max_features': 0.1, 'max_samples': 0.8402191

Fold 1 IBS: 0.23905204688392212
Fold 2 IBS: 0.2187928517163343
Fold 3 IBS: 0.2102452041922602
Fold 4 IBS: 0.22890896945323644
Fold 5 IBS: 0.21433456345556318
[I 2024-04-18 15:17:49,789] Trial 60 finished with value: 0.22226672714026327 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 11, 'min_samples_leaf': 11, 'max_depth': 16, 'n_estimators': 74, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.9528336631358474, 'min_weight_fraction_leaf': 0.23484468684942653}. Best is trial 6 with value: 0.21817221600733724.
Fold 1 IBS: 0.23541239037810044
Fold 2 IBS: 0.2157600681640186
Fold 3 IBS: 0.20665073702737224
Fold 4 IBS: 0.22705259747647938
Fold 5 IBS: 0.2114858383168534
[I 2024-04-18 15:17:55,160] Trial 61 finished with value: 0.21927232627256482 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 8, 'min_samples_leaf': 7, 'max_depth': 9, 'n_estimators': 397, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.6

Fold 1 IBS: 0.23749669116970254
Fold 2 IBS: 0.21417891364052918
Fold 3 IBS: 0.2066472918191204
Fold 4 IBS: 0.22577332010612122
Fold 5 IBS: 0.20791194308464916
[I 2024-04-18 15:19:20,271] Trial 75 finished with value: 0.2184016319640245 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 18, 'min_samples_leaf': 8, 'max_depth': 6, 'n_estimators': 346, 'oob_score': True, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.6413926717278569, 'min_weight_fraction_leaf': 0.0007388507314896667}. Best is trial 6 with value: 0.21817221600733724.
Fold 1 IBS: 0.23958121408927258
Fold 2 IBS: 0.21329051656876732
Fold 3 IBS: 0.20685348469946407
Fold 4 IBS: 0.2243628744610485
Fold 5 IBS: 0.2076885994354418
[I 2024-04-18 15:19:26,337] Trial 76 finished with value: 0.21835533785079883 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 18, 'min_samples_leaf': 4, 'max_depth': 5, 'n_estimators': 356, 'oob_score': True, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.6

Fold 1 IBS: 0.23843719064657817
Fold 2 IBS: 0.22064066243477068
Fold 3 IBS: 0.216117076423818
Fold 4 IBS: 0.23057174337958733
Fold 5 IBS: 0.21778600564999445
[I 2024-04-18 15:20:38,175] Trial 90 finished with value: 0.22471053570694974 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 14, 'min_samples_leaf': 4, 'max_depth': 2, 'n_estimators': 274, 'oob_score': True, 'warm_start': False, 'max_features': 0.1, 'max_samples': 0.42143690848778903, 'min_weight_fraction_leaf': 0.05305541913345835}. Best is trial 83 with value: 0.21749549175321342.
Fold 1 IBS: 0.23636584853089593
Fold 2 IBS: 0.2160409129934387
Fold 3 IBS: 0.20518796152371307
Fold 4 IBS: 0.22686151762465062
Fold 5 IBS: 0.21138354158709388
[I 2024-04-18 15:20:43,146] Trial 91 finished with value: 0.21916795645195847 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 17, 'min_samples_leaf': 4, 'max_depth': 4, 'n_estimators': 307, 'oob_score': True, 'warm_start': False, 'max_features': 'auto', 'max_samples': 0.378

In [64]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [65]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.791
train_ibs:  0.217


#### Test

In [66]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [67]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=13, max_features=None, max_leaf_nodes=16,
                   max_samples=0.8020091028727125, min_samples_leaf=1,
                   min_samples_split=5,
                   min_weight_fraction_leaf=0.002435943155287043,
                   n_estimators=277, random_state=123, warm_start=True)

C-index score: 0.584


ExtraSurvivalTrees(max_depth=6, max_features='auto', max_leaf_nodes=16,
                   max_samples=0.44903497421134486, min_samples_leaf=4,
                   min_samples_split=9,
                   min_weight_fraction_leaf=0.018023565445760347,
                   n_estimators=351, oob_score=True, random_state=123)

IBS: 0.223


In [68]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [69]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-18 15:21:20,989] A new study created in memory with name: no-name-c762fb50-0b55-44f9-b26b-e074962e3a33


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-18 15:21:45,037] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-18 15:21:59,450] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-18 15:29:26,477] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 12 with value: 0.6059172111077464.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-18 15:30:34,567] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'squar

Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-18 15:43:04,077] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.7565257046780437, 'learning_rate': 0.01188344684416992, 'dropout_rate': 0.3433523077110169, 'n_estimators': 318, 'criterion': 'squared_error', 'ccp_alpha': 2.063559212730723, 'min_weight_fraction_leaf': 0.25401273903421573, 'max_features': 'auto', 'min_impurity_decrease': 5.773435946665558e-07, 'validation_fraction': 0.8062268184400477, 'min_samples_split': 16, 'max_leaf_nodes': 18, 'min_samples_leaf': 5, 'max_depth': 3}. Best is trial 22 with value: 0.6190116704947725.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-18 15:44:32,845] Trial 26 finished with value: 0.5 and parameters: {'subsample': 0.8922404482621683, 'learning_rate': 0.011585260292674536, 'dropout_rate': 0.2527000999648632, 'n_estimators': 446, 'criterion': 'friedman_mse', 'ccp_alpha': 0.11820935501930148, 'min_weight_fraction

...

Fold 3 IBS: 0.22898186806977708
Fold 4 IBS: 0.24197477145927118
Fold 5 IBS: 0.2293955930480925
[I 2024-04-18 21:07:17,374] Trial 44 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9479336661285427, 'learning_rate': 0.0149361535238727, 'dropout_rate': 0.25335630680617827, 'n_estimators': 238, 'criterion': 'squared_error', 'ccp_alpha': 0.5490150526266808, 'min_weight_fraction_leaf': 0.4126953497238681, 'max_features': 'auto', 'min_impurity_decrease': 0.00078866474186123, 'validation_fraction': 0.9438667820963776, 'min_samples_split': 19, 'max_leaf_nodes': 19, 'min_samples_leaf': 17, 'max_depth': 3}. Best is trial 42 with value: 0.2339713101479977.
Fold 1 IBS: 0.24678393645753394
Fold 2 IBS: 0.23045362112354736
Fold 3 IBS: 0.22798640772703402
Fold 4 IBS: 0.24133177588235574
Fold 5 IBS: 0.22819468131537443
[I 2024-04-18 21:07:45,006] Trial 45 finished with value: 0.2349500845011691 and parameters: {'subsample': 0.5852424762732177, 'learning_rate': 0.065638506610498

Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-18 21:11:18,360] Trial 55 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.804828042321909, 'learning_rate': 0.023646082998228058, 'dropout_rate': 0.23926982708415356, 'n_estimators': 162, 'criterion': 'squared_error', 'ccp_alpha': 0.40074886285106287, 'min_weight_fraction_leaf': 0.2779066644542128, 'max_features': 'sqrt', 'min_impurity_decrease': 0.008146563872249941, 'validation_fraction': 0.8868940629916056, 'min_samples_split': 4, 'max_leaf_nodes': 16, 'min_samples_leaf': 19, 'max_depth': 4}. Best is trial 42 with value: 0.2339713101479977.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-18 21:12:14,552] Trial 56 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.6106355356441384, 'learning_rate': 0.0175381503002972, 'dropout_rate': 0.184039934

Fold 5 IBS: 0.22939559304809248
[I 2024-04-18 21:17:40,977] Trial 66 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.7279386202897999, 'learning_rate': 0.09984676062907398, 'dropout_rate': 0.12777796849617917, 'n_estimators': 154, 'criterion': 'squared_error', 'ccp_alpha': 1.4111498627316026, 'min_weight_fraction_leaf': 0.23517339076530247, 'max_features': 1, 'min_impurity_decrease': 0.00014974488025406402, 'validation_fraction': 0.6772907402354433, 'min_samples_split': 8, 'max_leaf_nodes': 16, 'min_samples_leaf': 20, 'max_depth': 3}. Best is trial 65 with value: 0.2330518973544506.
Fold 1 IBS: 0.24724710044658996
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977708
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-18 21:17:46,449] Trial 67 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.8919877355885673, 'learning_rate': 0.08526801927485488, 'dropout_rate': 0.1968183597819077, 'n_estimators': 45, 'cri

Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.2419747714592711
Fold 5 IBS: 0.2293955930480925
[I 2024-04-18 21:53:15,791] Trial 78 finished with value: 0.2359278435123307 and parameters: {'subsample': 0.9982285661999424, 'learning_rate': 0.09191504218778693, 'dropout_rate': 0.15698726385072695, 'n_estimators': 197, 'criterion': 'squared_error', 'ccp_alpha': 0.24807516981478814, 'min_weight_fraction_leaf': 0.20156232220196313, 'max_features': 0.1, 'min_impurity_decrease': 1.657879625668623e-05, 'validation_fraction': 0.743279291715464, 'min_samples_split': 6, 'max_leaf_nodes': 20, 'min_samples_leaf': 20, 'max_depth': 11}. Best is trial 75 with value: 0.23239681107011897.
Fold 1 IBS: 0.24690614034739003
Fold 2 IBS: 0.23156150181893284
Fold 3 IBS: 0.22844484931909595
Fold 4 IBS: 0.24167312234765576
Fold 5 IBS: 0.22876368497640898
[I 2024-04-18 21:53:24,052] Trial 79 finished with value: 0.23546985976189672 and parameters: {'s

Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-18 21:57:31,101] Trial 89 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9483409389571185, 'learning_rate': 0.08684995291290751, 'dropout_rate': 0.261199231115212, 'n_estimators': 33, 'criterion': 'squared_error', 'ccp_alpha': 0.262026471967215, 'min_weight_fraction_leaf': 0.30628737845191195, 'max_features': 0.1, 'min_impurity_decrease': 6.315494653924259e-06, 'validation_fraction': 0.7881463591488752, 'min_samples_split': 10, 'max_leaf_nodes': 19, 'min_samples_leaf': 15, 'max_depth': 4}. Best is trial 75 with value: 0.23239681107011897.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.22939559304809248
[I 2024-04-18 21:57:56,662] Trial 90 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9683729112775776, 'le

In [70]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [71]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.642
train_ibs:  0.232


#### Test

In [72]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [73]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.0019473494055538935,
                                 criterion='squared_error',
                                 dropout_rate=0.43613571687055647,
                                 learning_rate=0.09542136158814303,
                                 max_depth=20, max_features='sqrt',
                                 max_leaf_nodes=20,
                                 min_impurity_decrease=5.255459802548745e-07,
                                 min_samples_leaf=15, min_samples_split=10,
                                 min_weight_fraction_leaf=0.36743918461462793,
                                 n_estimators=66, random_state=123,
                                 subsample=0.569175138136116,
                                 validation_fraction=0.7087056989518657)

C-index score: 0.669


GradientBoostingSurvivalAnalysis(ccp_alpha=0.008911991304216094,
                                 criterion='squared_error',
                                 dropout_rate=0.1333613034046263,
                                 learning_rate=0.09420039979456486,
                                 max_depth=14, max_features=0.1,
                                 max_leaf_nodes=19,
                                 min_impurity_decrease=2.4912576060277585e-06,
                                 min_samples_leaf=17, min_samples_split=7,
                                 min_weight_fraction_leaf=0.2444025476799629,
                                 n_estimators=141, random_state=123,
                                 subsample=0.9998829056403309,
                                 validation_fraction=0.6909543967814417)

IBS: 0.224


In [74]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [75]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [76]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-18 22:00:04,159] A new study created in memory with name: no-name-823aea2a-1111-461e-a63b-5e50f3fc9011


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5358565737051793
Fold 2 C-index: 0.4903100775193798
Fold 3 C-index: 0.5553191489361702
Fold 4 C-index: 0.5057034220532319
Fold 5 C-index: 0.6201716738197425
[I 2024-04-18 22:00:08,691] Trial 0 finished with value: 0.5414721792067407 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.5414721792067407.
Fold 1 C-index: 0.5358565737051793
Fold 2 C-index: 0.48643410852713176
Fold 3 C-index: 0.5553191489361702
Fold 4 C-index: 0.49809885931558934
Fold 5 C-index: 0.6244635193133047
[I 2024-04-18 22:00:45,543] Trial 1 finished with value: 0.540034441959475 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 0 with value: 0.5414721792067407.
Fold 1 C-index: 0.599601593625498
Fold 2 C-index: 0.49806201550387597
Fold 3 C-index: 0.5595744680851064
Fold

Fold 1 C-index: 0.6215139442231076
Fold 2 C-index: 0.5368217054263565
Fold 3 C-index: 0.5957446808510638
Fold 4 C-index: 0.5513307984790875
Fold 5 C-index: 0.6523605150214592
[I 2024-04-18 22:06:01,796] Trial 19 finished with value: 0.5915543288002149 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.7254192287154788, 'n_estimators': 117, 'learning_rate': 0.09614402133777997}. Best is trial 12 with value: 0.5979858924640695.
Fold 1 C-index: 0.6215139442231076
Fold 2 C-index: 0.5193798449612403
Fold 3 C-index: 0.5872340425531914
Fold 4 C-index: 0.5703422053231939
Fold 5 C-index: 0.648068669527897
[I 2024-04-18 22:06:36,305] Trial 20 finished with value: 0.589307741317726 and parameters: {'subsample': 0.2630057481431337, 'dropout_rate': 0.18040218016888274, 'n_estimators': 423, 'learning_rate': 0.07788582119853761}. Best is trial 12 with value: 0.5979858924640695.
Fold 1 C-index: 0.6215139442231076
Fold 2 C-index: 0.5387596899224806
Fold 3 C-index: 0.6
Fold 4 C-index: 

Fold 1 C-index: 0.599601593625498
Fold 2 C-index: 0.5058139534883721
Fold 3 C-index: 0.5553191489361702
Fold 4 C-index: 0.5475285171102662
Fold 5 C-index: 0.6244635193133047
[I 2024-04-18 22:13:08,471] Trial 38 finished with value: 0.5665453464947222 and parameters: {'subsample': 0.597471300203124, 'dropout_rate': 0.15312457379192773, 'n_estimators': 37, 'learning_rate': 0.07288612365177065}. Best is trial 37 with value: 0.63135676757979.
Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.5271317829457365
Fold 3 C-index: 0.5702127659574469
Fold 4 C-index: 0.5779467680608364
Fold 5 C-index: 0.648068669527897
[I 2024-04-18 22:13:15,334] Trial 39 finished with value: 0.5881779733940009 and parameters: {'subsample': 0.17215875631487781, 'dropout_rate': 0.6293987146215918, 'n_estimators': 145, 'learning_rate': 0.08371618852440825}. Best is trial 37 with value: 0.63135676757979.
Fold 1 C-index: 0.6095617529880478
Fold 2 C-index: 0.5155038759689923
Fold 3 C-index: 0.5829787234042553
Fold 4 

...

Fold 2 IBS: 0.2437305585877217
Fold 3 IBS: 0.21828441799894427
Fold 4 IBS: 0.2799082411268824
Fold 5 IBS: 0.21959930006866285
[I 2024-04-18 23:31:14,790] Trial 19 finished with value: 0.24111557240242817 and parameters: {'subsample': 0.8350722272141606, 'dropout_rate': 0.8386110478636641, 'n_estimators': 135, 'learning_rate': 0.010447042153984048}. Best is trial 10 with value: 0.23589355598382986.
Fold 1 IBS: 0.24417441965516398
Fold 2 IBS: 0.24564410329045724
Fold 3 IBS: 0.21829558699624546
Fold 4 IBS: 0.2845284621600695
Fold 5 IBS: 0.21885186859083686
[I 2024-04-18 23:31:17,287] Trial 20 finished with value: 0.2422988881385546 and parameters: {'subsample': 0.9152086021979481, 'dropout_rate': 0.5638508730252081, 'n_estimators': 61, 'learning_rate': 0.025428571362532227}. Best is trial 10 with value: 0.23589355598382986.
Fold 1 IBS: 0.24431477640484145
Fold 2 IBS: 0.23832316205108614
Fold 3 IBS: 0.2199227079522707
Fold 4 IBS: 0.26746257866946227
Fold 5 IBS: 0.22278659465478365
[I 2024-

Fold 2 IBS: 0.3406669967514771
Fold 3 IBS: 0.29615252460210995
Fold 4 IBS: 0.3505624068968957
Fold 5 IBS: 0.23184246606960945
[I 2024-04-18 23:32:23,971] Trial 38 finished with value: 0.3040782160376305 and parameters: {'subsample': 0.708815775769025, 'dropout_rate': 0.10689409878635497, 'n_estimators': 159, 'learning_rate': 0.0802388198308747}. Best is trial 37 with value: 0.23588890155494274.
Fold 1 IBS: 0.25341663037436557
Fold 2 IBS: 0.2663073173733578
Fold 3 IBS: 0.24497687658347134
Fold 4 IBS: 0.2972154484829737
Fold 5 IBS: 0.20528323840152937
[I 2024-04-18 23:32:35,760] Trial 39 finished with value: 0.25343990224313956 and parameters: {'subsample': 0.36375004083294554, 'dropout_rate': 0.27268583127450263, 'n_estimators': 225, 'learning_rate': 0.021705015906159475}. Best is trial 37 with value: 0.23588890155494274.
Fold 1 IBS: 0.33133311685803907
Fold 2 IBS: 0.42148486722189815
Fold 3 IBS: 0.3206482058467423
Fold 4 IBS: 0.4376453570735614
Fold 5 IBS: 0.33886660696535603
[I 2024-0

Fold 2 IBS: 0.2342449746737762
Fold 3 IBS: 0.22345226599611065
Fold 4 IBS: 0.25187779626804524
Fold 5 IBS: 0.2258572613233101
[I 2024-04-18 23:34:04,336] Trial 57 finished with value: 0.23616255731631552 and parameters: {'subsample': 0.772073129968633, 'dropout_rate': 0.518660275195612, 'n_estimators': 102, 'learning_rate': 0.004505670695986405}. Best is trial 43 with value: 0.2358742101061661.
Fold 1 IBS: 0.24573008555077389
Fold 2 IBS: 0.25605624624517975
Fold 3 IBS: 0.22153729289324992
Fold 4 IBS: 0.2905183614616067
Fold 5 IBS: 0.21572267525691174
[I 2024-04-18 23:34:09,228] Trial 58 finished with value: 0.2459129322815444 and parameters: {'subsample': 0.8237089546855147, 'dropout_rate': 0.7587802984045586, 'n_estimators': 127, 'learning_rate': 0.018008460652548762}. Best is trial 43 with value: 0.2358742101061661.
Fold 1 IBS: 0.24965385253523084
Fold 2 IBS: 0.26782842078574676
Fold 3 IBS: 0.22744721139319166
Fold 4 IBS: 0.2973096991465234
Fold 5 IBS: 0.215052600743126
[I 2024-04-18

Fold 2 IBS: 0.2529774991711212
Fold 3 IBS: 0.21798102587116244
Fold 4 IBS: 0.28642688588297865
Fold 5 IBS: 0.20087748432833774
[I 2024-04-18 23:35:39,027] Trial 76 finished with value: 0.24045496078967651 and parameters: {'subsample': 0.13339093977454577, 'dropout_rate': 0.6259305242495848, 'n_estimators': 202, 'learning_rate': 0.022676239872402745}. Best is trial 75 with value: 0.23305706797082468.
Fold 1 IBS: 0.24182560868531242
Fold 2 IBS: 0.23430921754204745
Fold 3 IBS: 0.2151702621237061
Fold 4 IBS: 0.2684686047912394
Fold 5 IBS: 0.21317672567604404
[I 2024-04-18 23:35:43,429] Trial 77 finished with value: 0.23459008376366985 and parameters: {'subsample': 0.16428958972668722, 'dropout_rate': 0.6517748140985425, 'n_estimators': 114, 'learning_rate': 0.017289869718512596}. Best is trial 75 with value: 0.23305706797082468.
Fold 1 IBS: 0.2418289889586624
Fold 2 IBS: 0.2345753984919539
Fold 3 IBS: 0.2151340980306673
Fold 4 IBS: 0.2689894032140322
Fold 5 IBS: 0.2130069635490081
[I 2024-

Fold 1 IBS: 0.2406863202827842
Fold 2 IBS: 0.23719290727766623
Fold 3 IBS: 0.2139880463961989
Fold 4 IBS: 0.27307427524352845
Fold 5 IBS: 0.20542783875690165
[I 2024-04-18 23:37:34,384] Trial 95 finished with value: 0.2340738775914159 and parameters: {'subsample': 0.10055421726352921, 'dropout_rate': 0.6367468591731635, 'n_estimators': 156, 'learning_rate': 0.018320625747593914}. Best is trial 94 with value: 0.23230231719854308.
Fold 1 IBS: 0.24069365191467598
Fold 2 IBS: 0.23733770317855976
Fold 3 IBS: 0.21411919366368914
Fold 4 IBS: 0.2733885091832182
Fold 5 IBS: 0.20523810100634188
[I 2024-04-18 23:37:41,518] Trial 96 finished with value: 0.234155431789297 and parameters: {'subsample': 0.10430327789057646, 'dropout_rate': 0.7150842017502997, 'n_estimators': 156, 'learning_rate': 0.01849431229569403}. Best is trial 94 with value: 0.23230231719854308.
Fold 1 IBS: 0.2423826523606692
Fold 2 IBS: 0.24154153546303617
Fold 3 IBS: 0.21418809053102322
Fold 4 IBS: 0.27998173014531524
Fold 5 I

In [77]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [78]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.631
train_ibs:  0.232


#### Test

In [79]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [80]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.6198740536882945,
                                              learning_rate=0.08398598098613289,
                                              n_estimators=1, random_state=123,
                                              subsample=0.1424992712803435)

C-index score: 0.52


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.641812724307286,
                                              learning_rate=0.01230597296655908,
                                              n_estimators=110,
                                              random_state=123,
                                              subsample=0.10168659289889766)

IBS: 0.235


In [81]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

## Results

In [82]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
ExtraSurvivalTrees,0.791,1.0
Randomsurvivalforest,0.760,2.0
GradientBoosting,0.642,3.0
CoxElastic,0.638,4.0
CoxPH,0.637,5.0
CoxLasso,0.636,6.0
ComponentwiseGradientBoosting,0.631,7.0
CoxRidge,0.571,8.0


In [83]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
ExtraSurvivalTrees,0.217,1.0
Randomsurvivalforest,0.223,2.0
GradientBoosting,0.232,3.5
ComponentwiseGradientBoosting,0.232,3.5
CoxElastic,0.233,5.0
CoxRidge,0.236,6.0
CoxPH,0.240,7.5
CoxLasso,0.240,7.5


In [84]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
GradientBoosting,0.669,1.0
Randomsurvivalforest,0.633,2.0
ExtraSurvivalTrees,0.584,3.0
CoxLasso,0.566,4.0
CoxPH,0.563,5.0
CoxRidge,0.532,6.0
CoxElastic,0.524,7.0
ComponentwiseGradientBoosting,0.520,8.0


In [85]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs

,IBS,rank
Randomsurvivalforest,0.216,1.0
ExtraSurvivalTrees,0.223,2.0
GradientBoosting,0.224,3.0
CoxRidge,0.229,4.0
CoxElastic,0.235,5.5
ComponentwiseGradientBoosting,0.235,5.5
CoxLasso,0.288,7.0
CoxPH,0.291,8.0


In [86]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  
file_path = 'path_to_your_folder/'  

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d1/dfs/standard/no_selection/'  

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d1_dfs_standard_no_selection_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [87]:
from datetime import date
today = date.today()
print("Date: ", today)

Date:  2024-04-18
